In [1]:
import psutil
import shutil

# RAM
ram = psutil.virtual_memory()

print("=== SYSTEM RAM ===")
print(f"Total RAM     : {ram.total / (1024**3):.2f} GB")
print(f"Available RAM : {ram.available / (1024**3):.2f} GB")
print(f"Used RAM      : {ram.used / (1024**3):.2f} GB")

# Disk
total, used, free = shutil.disk_usage("/")

print("\n=== SYSTEM DISK ===")
print(f"Total Disk : {total / (1024**3):.2f} GB")
print(f"Used Disk  : {used / (1024**3):.2f} GB")
print(f"Free Disk  : {free / (1024**3):.2f} GB")

=== SYSTEM RAM ===
Total RAM     : 499.07 GB
Available RAM : 470.38 GB
Used RAM      : 28.69 GB

=== SYSTEM DISK ===
Total Disk : 435.54 GB
Used Disk  : 220.47 GB
Free Disk  : 192.88 GB


In [2]:
!nvidia-smi
!df -h ~
!free -h

Thu Oct 22 21:32:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          Off |   00000000:21:00.0 Off |                    0 |
| N/A   63C    P0            253W /  250W |    9845MiB /  40960MiB |    100%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
!df -h

Filesystem                         Size  Used Avail Use% Mounted on
overlay                            436G  221G  193G  54% /
tmpfs                               64M     0   64M   0% /dev
/dev/rbd4                           49G   44G  5.4G  90% /home
tmpfs                              100G     0  100G   0% /dev/shm
/dev/mapper/ubuntu--vg-ubuntu--lv  436G  221G  193G  54% /etc/hosts
tmpfs                              100G  4.5M  100G   1% /run/nvidia-persistenced/socket
tmpfs                              250G     0  250G   0% /proc/acpi
tmpfs                              250G     0  250G   0% /proc/scsi
tmpfs                              250G     0  250G   0% /sys/firmware
tmpfs                              250G     0  250G   0% /sys/devices/virtual/powercap


In [4]:
import os
os.makedirs("models", exist_ok=True)
with open(".gitignore", "w") as f:
    f.write("models/\n")

In [5]:
!pip install -U pip huggingface_hub transformers accelerate

  Using cached huggingface_hub-2.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached transformers-5.20.2-py3-none-any.whl.metadata (31 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.8 kB)
Using cached huggingface_hub-2.2.0-py3-none-any.whl (839 kB)
Using cached transformers-5.20.2-py3-none-any.whl (12.8 MB)
Using cached tokenizers-0.23.3-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.4 MB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers━━━━━━━━━━━━━━ 0/3 [huggingface_hub]
    Found existing installation: tokenizers 0.21.40/3 [huggingface_hub]
    Uninstalling tokenizers-0.21.4:━━━━━━━━━ 0/3 [huggingface_hub]
      Successfully uninstalled tokenizers-0.21.4 0/3 [huggingface_hub]
  Attempting uninstall: transformers━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
%cd ~/Team-Horizon
!pip install -q "transformers==4.51.3"

from huggingface_hub import snapshot_download

MODELS = {
    "qwen2.5-coder-7b-awq": "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ",
    "deepseek-coder-v2-lite-awq": "TechxGenus/DeepSeek-Coder-V2-Lite-Instruct-AWQ",
    # "glm-9b-PLACEHOLDER": "THUDM/glm-4-9b",
}

/home/nihkhe/Team-Horizon


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("loading Qwen2.5-Coder-7B...")
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct", torch_dtype=torch.bfloat16, device_map="cuda:0")
inputs = tok("2+2=", return_tensors="pt").to("cuda:0")
output = model.generate(**inputs, max_new_tokens=10)
print("DONE:", tok.decode(output[0], skip_special_tokens=True))

del model, tok
torch.cuda.empty_cache()
print("memory cleared")

loading Qwen2.5-Coder-7B...


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

DONE: 2+2=4

This is a basic mathematical equation that demonstrates
memory cleared


In [7]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 65.8 MB/s  0:00:00 eta 0:00:01


In [8]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.cache_utils import DynamicCache
DynamicCache.get_max_length = lambda self: None

quant_config = BitsAndBytesConfig(load_in_8bit=True)

print("loading DeepSeek-Coder-V2-Lite (8-bit, single GPU)...")
tok = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct",
    quantization_config=quant_config,
    device_map="cuda:1",   # switched from cuda:0, since that one's seen the most reuse tonight
    trust_remote_code=True
)
inputs = tok("2+2=", return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=10)
print("DONE:", tok.decode(output[0], skip_special_tokens=True))

loading DeepSeek-Coder-V2-Lite (8-bit, single GPU)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


DONE: 2+2=4

2+3=5


